In [5]:
import networkx as nx
import pandas as pd
import random

In [6]:
G = nx.read_edgelist(
    "Wiki-Vote.txt",
    comments="#",
    nodetype=int,
    create_using=nx.DiGraph()
)

#### Generate positive edges

Positive edges are generated by selecting a random sample of edges from the graph excluding bridges. Only by removing a bridge would you disconnect a weakly connected graph.

In [7]:
bridges = set(nx.bridges(G.to_undirected()))

non_bridges = [
    (u, v) for (u, v) in G.edges()
    if (u, v) not in bridges and (v, u) not in bridges
]

positive_edges = random.sample(non_bridges, int(0.1 * G.number_of_edges()))

H = G.copy() # Transfomed graph without positve edges
H.remove_edges_from(positive_edges)

#### Generate negative edges

Negative edges are edges (u, v) where a path v to u exists but not a path from u to v.

In [8]:
negative_edges = []
nodes = list(G.nodes())

while len(negative_edges) < len(positive_edges):
    u, v = random.sample(nodes, 2)

    if nx.has_path(G, v, u) and not nx.has_path(G, u, v):
        negative_edges.append((u, v))

#### Labeled dataset for link predictrion

In the datset positive edges are labeled with 1 and negative edges with 0.

In [9]:
data = []

for u, v in positive_edges:
    data.append((u, v, 1))

for u, v in negative_edges:
    data.append((u, v, 0))

df = pd.DataFrame(data)
df.to_csv("link_prediction_dataset.csv", index=False, header=False)